In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import re

# 1. Mount Google Drive to access the dataset
# Drive is mounted manually

# Define the input path provided
file_path = '/content/drive/MyDrive/DAV_Project/five-years-11.xlsx'

# 2. Read all sheets from the Excel file
print(f"Loading Excel file from: {file_path}")
# sheet_name=None loads all sheets into a dictionary: { 'SheetName': DataFrame }
xls = pd.read_excel(file_path, sheet_name=None)
print(f"Found {len(xls)} sheets. Processing...")

all_dfs = []

# 3. Define the cleaning function for a single sheet
def clean_pama_sheet(df, sheet_name):
    # Drop completely empty rows and columns
    df = df.dropna(how='all').dropna(axis=1, how='all')

    # Reset columns to integers for reliable positional indexing
    df.columns = list(range(df.shape[1]))

    # Find the Metric Column ('Prod.' or 'Sale')
    try:
        metric_col_idx = df.apply(
            lambda x: x.astype(str).str.contains(r'^(Prod\.?|Sale)$', flags=re.IGNORECASE, na=False).sum()
        ).idxmax()
    except ValueError:
        return None # Skip sheet if no metric column is found

    brand_col_idx = metric_col_idx - 1

    # Forward-fill Brand/Model (since 'Sale' rows usually have a blank brand name)
    df['Brand_Model'] = df[brand_col_idx].ffill()

    # Filter strictly for 'Prod' or 'Sale' rows
    df_filtered = df[
        df[metric_col_idx].astype(str).str.contains(r'^(Prod\.?|Sale)$', flags=re.IGNORECASE, na=False)
    ].copy()

    # Standardize the metric name
    df_filtered['Metric_Type'] = df_filtered[metric_col_idx].astype(str).str.replace(
        r'Prod\.', 'Prod', regex=True, flags=re.IGNORECASE
    ).str.title()

    # Identify the 12 Month columns directly to the right of the metric column
    potential_month_cols = [c for c in df.columns if isinstance(c, int) and c > metric_col_idx]
    valid_month_cols = [c for c in potential_month_cols if df_filtered[c].notna().sum() > 0]

    # Ensure we grab exactly the 12 months (July to June)
    month_cols = valid_month_cols[:12]
    if len(month_cols) < 12:
        return None

    df_final = df_filtered[['Brand_Model', 'Metric_Type'] + month_cols].copy()

    # Rename columns to standard Month_1 through Month_12
    month_names = [f'Month_{i}' for i in range(1, 13)]
    df_final.columns = ['Brand_Model', 'Metric_Type'] + month_names

    # Extract Fiscal Year from the sheet name (e.g., '2024-25')
    fy_match = re.search(r'20\d{2}-\d{2}', str(sheet_name))
    fiscal_year = fy_match.group() if fy_match else sheet_name
    df_final.insert(0, 'Fiscal_Year', fiscal_year)

    # Melt to Long Format (Tidy Data)
    df_melted = df_final.melt(
        id_vars=['Fiscal_Year', 'Brand_Model', 'Metric_Type'],
        value_vars=month_names,
        var_name='Month_Index',
        value_name='Units'
    )
    df_melted['Month_Index'] = df_melted['Month_Index'].str.replace('Month_', '').astype(int)

    return df_melted

# Apply cleaning to all sheets
for sheet_name, df in xls.items():
    cleaned_df = clean_pama_sheet(df, sheet_name)
    if cleaned_df is not None:
        all_dfs.append(cleaned_df)

# 4. Concatenate into a Master DataFrame
master_df = pd.concat(all_dfs, ignore_index=True)

# Clean up 'Units' (Remove commas, dashes, convert to integers)
master_df['Units'] = master_df['Units'].astype(str).str.replace(',', '', regex=False).str.strip()
master_df['Units'] = pd.to_numeric(master_df['Units'], errors='coerce').fillna(0).astype(int)

# 5. Generate absolute DateTimes for Time-Series Modeling
def generate_date(row):
    try:
        # Extract starting year from fiscal year string (e.g., '2023-24' -> 2023)
        year_str = str(row['Fiscal_Year']).strip()
        parts = re.split(r'[-/]', year_str)
        if len(parts) != 2: return pd.NaT

        y1 = int('20' + parts[0][-2:]) if len(parts[0]) == 2 else int(parts[0])

        # Month 1-6 = July-Dec of starting year | Month 7-12 = Jan-Jun of next year
        if row['Month_Index'] <= 6:
            actual_month = row['Month_Index'] + 6
            actual_year = y1
        else:
            actual_month = row['Month_Index'] - 6
            actual_year = y1 + 1

        return pd.to_datetime(f"{actual_year}-{actual_month:02d}-01")
    except Exception:
        return pd.NaT

master_df['Date'] = master_df.apply(generate_date, axis=1)

# 6. Final Polish and Export back to Drive
# Drop rows that are just PAMA subtotals and drop invalid dates
master_df = master_df[~master_df['Brand_Model'].astype(str).str.contains('Total|TOTAL', na=False, regex=True)]
master_df = master_df.dropna(subset=['Date'])

# Reorder columns
master_df = master_df[['Date', 'Fiscal_Year', 'Brand_Model', 'Metric_Type', 'Units']]
master_df = master_df.sort_values(by=['Brand_Model', 'Date', 'Metric_Type']).reset_index(drop=True)

# Save the final CSV to the same directory
output_path = '/content/drive/MyDrive/DAV_Project/01_pama_cleaned_long_format.csv'
master_df.to_csv(output_path, index=False)

print(f"Data Engineering Complete! Data saved to: {output_path}")
print(f"Total Rows: {len(master_df)}")
display(master_df.head(10))

Loading Excel file from: /content/drive/MyDrive/DAV_Project/five-years-11.xlsx


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


Found 18 sheets. Processing...


/tmp/ipykernel_4769/2098884800.py:31: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  lambda x: x.astype(str).str.contains(r'^(Prod\.?|Sale)$', flags=re.IGNORECASE, na=False).sum()
/tmp/ipykernel_4769/2098884800.py:31: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  lambda x: x.astype(str).str.contains(r'^(Prod\.?|Sale)$', flags=re.IGNORECASE, na=False).sum()
/tmp/ipykernel_4769/2098884800.py:31: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  lambda x: x.astype(str).str.contains(r'^(Prod\.?|Sale)$', flags=re.IGNORECASE, na=False).sum()
/tmp/ipykernel_4769/2098884800.py:31: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  lambda x: x.astype(str)

Data Engineering Complete! Data saved to: /content/drive/MyDrive/DAV_Project/01_pama_cleaned_long_format.csv
Total Rows: 17736


,Date,Fiscal_Year,Brand_Model,Metric_Type,Units
0,2021-07-01,2021-22,BAIC BJ40L,Prod,6
1,2021-07-01,2021-22,BAIC BJ40L,Sale,0
2,2021-08-01,2021-22,BAIC BJ40L,Prod,6
3,2021-08-01,2021-22,BAIC BJ40L,Sale,1
4,2021-09-01,2021-22,BAIC BJ40L,Prod,14
5,2021-09-01,2021-22,BAIC BJ40L,Sale,3
6,2021-10-01,2021-22,BAIC BJ40L,Prod,9
7,2021-10-01,2021-22,BAIC BJ40L,Sale,21
8,2021-11-01,2021-22,BAIC BJ40L,Prod,1
9,2021-11-01,2021-22,BAIC BJ40L,Sale,3
